# PrepareData -- Experian / Test / NEW

Split out from the original `PrepareDataNewMethod.ipynb` so this specific
(bureau, split, method) combo runs independently with its own log file.

**Why split:** the combined notebook finished `equifax/train` and `experian/train`
(both phases) for both methods, then stalled inside `transunion/train` Phase 1
(NEW stopped at chunk 463/534, OLD at 444/534). Splitting per (bureau, split,
method) means each combo can be restarted on its own without re-running the
already-done pieces, and each run has its own log.

**This notebook:** runs the **NEW** aggregator over `experian` / `test` only.

The first thing it does is **clear `normalized_new/` and `processed_new/`
for this (bureau, split)** so a half-finished run doesn't leak into the new run.

**Outputs:**
- `payment_processing_research_data/experian/test/normalized_new/part-NNNNN.parquet`
- `payment_processing_research_data/experian/test/processed_new/part-NNN.parquet`
- Log: `<cwd>/process_experian_test_NEW.log`


In [ ]:
%pip install -e /home/jag/model-engine

In [ ]:
%pip install -e /home/jag/feature-engine-parts-processor-change

In [ ]:
import sys
sys.path.insert(0, '/home/jag/feature-engine-parts-processor-change')
sys.path.insert(0, '/home/jag/model-engine')

import model_engine, feature_engine_parts
print('model_engine        :', model_engine.__file__)
print('feature_engine_parts:', feature_engine_parts.__file__)
# Sanity: both should resolve to /home/jag/... (the editable checkouts).

In [ ]:
from model_engine.assets.utils import load_asset
asset = load_asset('experian/arf7/fe2/trade.json')
agg_step = next(s for s in asset['preprocess'] if s['type'] == 'PaymentPatternsAggregatorV2')
print('aggregator params keys:', sorted(agg_step['params'].keys()))
assert 'missing_data_chars' in agg_step['params'], (
    'asset must have top-level missing_data_chars for the NEW aggregator. '
    'Check that model_engine is loading from /home/jag/model-engine/, not ~/.local/.')
print('missing_data_chars:', agg_step['params']['missing_data_chars'])


In [ ]:
import gc, os, sys, warnings
from pathlib import Path
import pandas as pd

from model_engine.feature_engine_V2.listed_objects_engines import MapperV2, PreprocessorV2
from model_engine.feature_engine_V2.feature_engine    import AggregationEngine
from configs import EXPERIAN, unmapped_dir, normalized_dir_new, processed_dir_new

warnings.filterwarnings('ignore')

BUREAU = EXPERIAN
SPLIT = 'test'
N_BUCKETS = 100

mapper       = MapperV2(api=asset['mapping'])
preprocessor = PreprocessorV2(api=asset['preprocess'])
agg_asset    = load_asset('aggregation/fe2/trade.json')
agg_eng      = AggregationEngine(asset=agg_asset, table_name='trade')
print(f'MapperV2 ({len(asset["mapping"])} steps) + PreprocessorV2 ({len(asset["preprocess"])} steps) + AggregationEngine ready')


In [ ]:
class _Tee:
    def __init__(self, *streams):
        self.streams = streams
    def write(self, data):
        for s in self.streams:
            try:
                s.write(data)
                s.flush()
            except Exception:
                pass
    def flush(self):
        for s in self.streams:
            try: s.flush()
            except Exception: pass

def clear_dir(d):
    d = Path(d)
    if d.exists():
        for f in d.glob('*'):
            if f.is_file():
                f.unlink()
    d.mkdir(parents=True, exist_ok=True)

def process_one():
    cfg      = BUREAU
    bureau   = cfg['bureau']
    split    = SPLIT
    in_dir   = Path(unmapped_dir(cfg, split))
    norm_dir = Path(normalized_dir_new(cfg, split))
    proc_dir = Path(processed_dir_new(cfg, split))

    log_path = Path(os.getcwd()) / f'process_{bureau}_{split}_NEW.log'
    original_stdout = sys.stdout
    with open(log_path, 'w', buffering=1) as log_file:
        sys.stdout = _Tee(original_stdout, log_file)
        try:
            print(f'(logging to {log_path})')
            print(f'\n##### {bureau}/{split} / NEW #####')

            parts = sorted(in_dir.glob('part-*.parquet')) if in_dir.exists() else []
            if not parts:
                print(f'MISSING unmapped chunks at {in_dir}  -- run save_unmapped_data.ipynb first')
                return

            print(f'clearing normalized + processed dirs first (idempotent)')
            print(f'  rm {norm_dir}/*')
            print(f'  rm {proc_dir}/*')
            clear_dir(norm_dir)
            clear_dir(proc_dir)

            # PHASE 1 -- map + preprocess in row chunks
            print(f'\nphase 1: map + preprocess ({len(parts)} chunks) -> {norm_dir}')
            for i, in_path in enumerate(parts):
                chunk      = pd.read_parquet(in_path)
                mapped     = mapper.transform(chunk)
                normalized = preprocessor.transform(mapped)
                out_path   = norm_dir / f'part-{i:05d}.parquet'
                normalized.to_parquet(out_path, index=False)
                print(f'  saved {i + 1}/{len(parts)} normalized -> {out_path}')
                del chunk, mapped, normalized
                gc.collect()

            # PHASE 2 -- aggregate per ZEST_KEY hash bucket
            print(f'\nphase 2: load full normalized, split into {N_BUCKETS} ZEST_KEY groups -> {proc_dir}')
            df = pd.read_parquet(norm_dir)
            print(f'  loaded {len(df):,} normalized rows')
            df['_bucket'] = (
                pd.util.hash_pandas_object(df['ZEST_KEY'], index=False) % N_BUCKETS
            ).astype('int16')

            total_rows = 0
            for b in range(N_BUCKETS):
                sub = df[df['_bucket'] == b].drop(columns='_bucket')
                if not len(sub):
                    continue
                processed = agg_eng.transform(sub)
                out_path = proc_dir / f'part-{b:03d}.parquet'
                processed.to_parquet(out_path, index=False)
                total_rows += len(processed)
                print(f'  saved group {b + 1}/{N_BUCKETS} processed ({total_rows:,} ZEST_KEYs so far) -> {out_path}')
                del sub, processed
                gc.collect()

            del df
            gc.collect()
            print(f'\ndone: {len(parts)} normalized chunks, {N_BUCKETS} processed groups, {total_rows:,} ZEST_KEYs')
        finally:
            sys.stdout = original_stdout
    print(f'process_one() done. Log saved to {log_path}')

process_one()
